# 05. Molecular Orbitals & Volumetric Charge Density Cubes

Quantum chemistry calculations give us two complementary views of where electrons live and what energies they hold:
- **Molecular orbital (MO) ladders** (parsed from Gaussian `.log` and ORCA `.out`) stack discrete orbital energies vertically. The **HOMO** (Highest Occupied Molecular Orbital) and **LUMO** (Lowest Unoccupied Molecular Orbital) sit right at the frontier of chemical reactivity—their separation, the **HOMO–LUMO gap**, is a quick indicator of kinetic stability and optical absorption energy.
- **3D volumetric grids** (parsed from Gaussian `.cube` and VASP `CHGCAR`/`LOCPOT`) store a scalar field—such as electron charge density or an orbital wavefunction $\psi(\mathbf{r})$—at every point on a 3D grid.

For 2D slices through a 3D grid, `VolumetricGrid.slice_mesh()` transforms grid indices along the exact voxel lattice vectors into an orthonormal 2D plane basis—keeping real-space geometry undistorted even for hexagonal, monoclinic, or triclinic cells.

In [ ]:
# Install xkcd-matchem automatically in Google Colab
import importlib.util


if importlib.util.find_spec("xkcd_matchem") is None:
    %pip install -q "git+https://github.com/vzordillo/xkcd-matchem.git"


## Basic Usage

Let's pair benzene's degenerate molecular orbital ladder (`benzene_orca.out`, automatically labeling the HOMO–LUMO gap) with a $z$-midplane slice of water's HOMO (`water_homo.cube`):

In [ ]:
import matplotlib.pyplot as plt

import xkcd_matchem as xm


benz = xm.load("benzene_orca.out")
cube = xm.load("water_homo.cube")

with xm.xkcd_style("classic"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.8, 5.0))
    xm.plot_mo_diagram(
        benz.metadata["orbitals"], ax=ax1, title="Benzene MOs (ORCA)"
    )
    xm.plot_volumetric_slice(
        cube, axis="z", ax=ax2, title="Water HOMO Slice (z-midplane)"
    )
    fig.tight_layout()

## Advanced Usage

You can inspect the underlying 2D projected coordinates and orthonormal plane basis directly via `cube.slice_mesh(axis=...)`, pick any Matplotlib diverging colormap (`cmap="BrBG"`, `"RdBu_r"`, `"PuOr"`), zoom into an orbital energy `window=(emin, emax)`, and apply custom color palettes:

In [ ]:
uu, vv, plane, basis = cube.slice_mesh(axis="y")
print("Orthonormal 2x3 slice plane basis:\n", basis)
print("Mesh shape:", plane.shape)

with xm.xkcd_style("notebook"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 4.8))
    xm.plot_volumetric_slice(
        cube,
        axis="x",
        cmap="BrBG",
        levels=11,
        preset="notebook",
        ax=ax1,
        title="Water HOMO (x-slice, BrBG)",
    )
    xm.plot_mo_diagram(
        benz.metadata["orbitals"],
        window=(-16.0, 4.0),
        palette=[
            "#2A9D8F", "#2A9D8F", "#2A9D8F", "#E76F51",
            "#E76F51", "#264653", "#E76F51",
        ],
        preset="notebook",
        ax=ax2,
        title="Custom MO Window & Earth Palette",
    )
    fig.tight_layout()

:::{tip}
**Try Next:** Pass `slice_index=...` to `plot_volumetric_slice()` to step through specific grid planes (for example, slicing directly through a surface adsorbate layer in a VASP `CHGCAR`), or jump to **Tutorial 06** for vibrational, electronic, and diffraction spectra.
:::